# EcoAcoustic Sentinel: pipeline completo en Google Colab

Este notebook ejecuta, sobre **todas** las grabaciones AudioMoth de CONDESAN guardadas en tu Google Drive, el mismo pipeline del informe (`docs/informe_finetuning`):

1. indexa las grabaciones y genera la **hoja de anotación aleatoria** (conjunto de prueba sin sesgo);
2. extrae los *embeddings* y puntuaciones de BirdNET V2.4 de cada ventana de 3 s;
3. entrena y evalúa los clasificadores (validación cruzada por sesión, LODO, bootstrap, curva de aprendizaje) y exporta el clasificador local;
4. genera figuras y tablas y empaqueta los resultados;
5. cuando la hoja esté anotada, hace la **evaluación sin sesgo**.

**Antes de empezar**

* *Entorno de ejecución → Cambiar tipo de entorno*: elige **CPU** (la GPU no acelera nada aquí: BirdNET TFLite y las cabezas de clasificación corren en CPU). Si tu plan lo ofrece, activa **RAM alta**; con el plan Ultra puedes usar la ejecución en segundo plano.
* Si la carpeta *AudioMoth - CONDESAN* aparece en **Compartido conmigo**, crea un acceso directo en **Mi unidad** (clic derecho → *Organizar* → *Añadir acceso directo*) y deja la ruta del paso 0 como está.
* **Seguridad de los datos**: el notebook solo **lee** la carpeta de audios. Todo lo que genera va a una carpeta nueva, `EcoRadar_trabajo`, en tu Drive. No se borra ni se modifica nada de la carpeta original.
* Si Colab se desconecta, vuelve a ejecutar los pasos 0 a 3 y continúa: la extracción del paso 5 retoma donde quedó.

Solo se procesan las grabaciones de las sesiones revisadas en TablaFinal (grabadora × día × periodo): fuera de ellas no hay etiquetas. El paso 3 muestra cuántas son. Tiempo orientativo con la CPU estándar de Colab: extracción, 2–4 s por grabación (la prueba del paso 3 sale más lenta porque incluye la carga del modelo); entrenamiento en modo rápido, 30–90 min.

## 0. Parámetros

In [ ]:
DATA_ROOT = "/content/drive/MyDrive/AudioMoth - CONDESAN"  # carpeta original (solo lectura)
WORK_DIR = "/content/drive/MyDrive/EcoRadar_trabajo"         # carpeta NUEVA para todo lo que se genera
REPO_URL = "https://github.com/Guivve-A/Aves_radar.git"
BRANCH = "main"
ANOTACION_DIR = f"{WORK_DIR}/anotacion_v3"  # carpeta de la hoja de anotacion
N_ANOTACION = 300    # ventanas de 3 s para la anotación exhaustiva
MODO_RAPIDO = True   # métodos principales; False reproduce todas las variantes del informe (más lento)

## 1. Conectar Google Drive y revisar los datos

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
from pathlib import Path

data_root, work_dir = Path(DATA_ROOT), Path(WORK_DIR)
assert data_root.is_dir(), f"No encuentro {DATA_ROOT}: revisa la ruta o crea el acceso directo en Mi unidad."
assert (data_root / "Resultados" / "TablaFinal.xlsx").exists(), "Falta Resultados/TablaFinal.xlsx en la carpeta de datos."
assert work_dir.resolve() != data_root.resolve() and data_root.resolve() not in work_dir.resolve().parents, \
    "WORK_DIR no puede estar dentro de la carpeta de audios."
work_dir.mkdir(parents=True, exist_ok=True)

n_wav, n_bytes = 0, 0
for dirpath, _, filenames in os.walk(data_root):
    for name in filenames:
        if name.lower().endswith(".wav"):
            n_wav += 1
            n_bytes += os.path.getsize(os.path.join(dirpath, name))
print(f"{n_wav} archivos WAV, {n_bytes / 1e9:.1f} GB en {DATA_ROOT}")

## 2. Código del proyecto y dependencias

Se instala solo lo que necesita el pipeline (`birdnet-analyzer`, `resampy`, `openpyxl`) sobre los paquetes de Colab (Python 3.13).
**No** instales `requirements.txt` de la app: fija versiones para Python 3.11 en Windows.

In [ ]:
import subprocess, sys

repo = Path("/content/Aves_radar")
if repo.exists():
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(repo)], check=True)
%cd /content/Aves_radar
!pip install -q "birdnet-analyzer==2.4.0" resampy openpyxl

import numpy, pandas, sklearn, tensorflow
print("Python", sys.version.split()[0], "| numpy", numpy.__version__, "| pandas", pandas.__version__,
      "| scikit-learn", sklearn.__version__, "| tensorflow", tensorflow.__version__)

## 3. Modelo BirdNET V2.4 y prueba con dos grabaciones

El modelo se toma del paquete `birdnetlib` de PyPI (mismo archivo `BirdNET_GLOBAL_6K_V2.4_Model_FP32.tflite` que usa la app).
Licencia de BirdNET: CC BY-NC-SA 4.0 (uso no comercial).

In [ ]:
import glob, zipfile

model_dir = Path("/content/birdnet_models/V2.4")
if not (model_dir / "BirdNET_GLOBAL_6K_V2.4_Model_FP32.tflite").exists():
    subprocess.run([sys.executable, "-m", "pip", "download", "-q", "--no-deps", "birdnetlib==0.18.1", "-d", "/content/wheels"], check=True)
    model_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(glob.glob("/content/wheels/birdnetlib-0.18.1-*.whl")[0]) as z:
        for name in z.namelist():
            if name.startswith("birdnetlib/models/analyzer/BirdNET_GLOBAL_6K_V2.4_"):
                (model_dir / Path(name).name).write_bytes(z.read(name))
print(sorted(p.name for p in model_dir.iterdir()))

os.environ["ECO_DATA_ROOT"] = DATA_ROOT
os.environ["ECO_WORK_DIR"] = WORK_DIR
os.environ["ECO_BIRDNET_MODEL_DIR"] = str(model_dir)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
THREADS = os.cpu_count() or 2

import shutil, time

def guardar_resultados(etiqueta, solo=None):
    """Copia a WORK_DIR un .zip con resultados agregados (nunca audio ni detecciones individuales)."""
    stamp = time.strftime("%Y%m%d_%H%M")
    staging = Path(f"/content/_paquete_{etiqueta}_{stamp}")
    results = Path("/content/Aves_radar/finetuning/results")
    if solo:
        (staging / "finetuning" / "results").mkdir(parents=True)
        for name in solo:
            if (results / name).exists():
                shutil.copy(results / name, staging / "finetuning" / "results" / name)
    else:
        shutil.copytree(results, staging / "finetuning" / "results")
        shutil.copytree("/content/Aves_radar/local_models/condesan", staging / "local_models" / "condesan")
        if (work_dir / "embeddings_errors.csv").exists():
            shutil.copy(work_dir / "embeddings_errors.csv", staging / "embeddings_errors.csv")
    archivo = shutil.make_archive(str(work_dir / f"resultados_{etiqueta}_{stamp}"), "zip", staging)
    print("Guardado en", archivo)

!python -m finetuning.drive_mirror index
import pandas as pd
index = pd.read_csv(work_dir / "mirror_index.csv")
print(index.groupby(["device", "period"]).size().unstack(fill_value=0))
from finetuning.drive_mirror import labeled_scope
scope = labeled_scope(index)
print(f"Extraccion estimada: {len(scope) * 3 / 3600:.1f} h (a ~3 s por grabacion)")

# Prueba: dos grabaciones (mide el tiempo por archivo antes del proceso largo).
!python -m finetuning.embeddings --limit 2 --threads {THREADS}

## 4. Hoja de anotación aleatoria

Elige `N_ANOTACION` ventanas de 3 s **al azar** (sin mirar BirdNET), repartidas por grabadora y periodo, y crea en `EcoRadar_trabajo/anotacion_v3/`:

* `anotacion.xlsx`: hoja *Anotacion* para rellenar, hoja *Especies* de referencia e *Instrucciones*;
* `clips/`: un WAV de 5 s por ventana (la ventana a anotar va del segundo 1 al 4).

El equipo debe anotar **todas** las especies audibles en cada ventana, con nombre científico separado por `;` (o `sin_aves = si`).
Si la hoja ya existe, este paso no la reemplaza, para no perder anotaciones.

In [ ]:
!python -m finetuning.sample_annotation --n {N_ANOTACION} --out "{ANOTACION_DIR}"

## 5. Embeddings de todas las grabaciones (paso largo)

Guarda un archivo por grabación en `EcoRadar_trabajo/embeddings/`. Si Colab se desconecta, ejecuta otra vez los pasos 0–3 y este: continúa donde quedó.
Los archivos que no se puedan leer se omiten y quedan listados en `embeddings_errors.csv`.

In [ ]:
!python -m finetuning.embeddings --threads {THREADS}

## 6. Entrenamiento, evaluación y exportación del clasificador local

In [ ]:
TRAIN_ARGS = "--rapido" if MODO_RAPIDO else ""
!python -m finetuning.train {TRAIN_ARGS}
guardar_resultados("entrenamiento")

## 7. Figuras, tablas y paquete final

Crea `EcoRadar_trabajo/resultados_informe_<fecha>.zip` con métricas agregadas, figuras, tablas y el clasificador local (sin audio ni detecciones individuales).
Cuando termine, avísame: lo leo desde tu Drive, actualizo el informe con el conjunto completo y lo subo al repositorio.

In [ ]:
!python -m finetuning.report
guardar_resultados("informe")

## 8. Evaluación sin sesgo (cuando la hoja esté anotada)

Puede ejecutarse con la anotación a medias: las filas vacías se ignoran.
Si el equipo convirtió la hoja a Google Sheets, descárgala como `.xlsx` (*Archivo → Descargar → Microsoft Excel*), súbela a `EcoRadar_trabajo/anotacion_v3/` y pon su nombre abajo.
En una sesión nueva de Colab, ejecuta antes los pasos 0–3 (las predicciones del paso 6 quedan guardadas en `EcoRadar_trabajo`). El resultado se guarda en `resultados_evaluacion_<fecha>.zip`.

In [ ]:
ANOTACION_RELLENADA = f"{ANOTACION_DIR}/anotacion.xlsx"
!python -m finetuning.evaluate_annotations --anotacion "{ANOTACION_RELLENADA}"
guardar_resultados("evaluacion", solo=["annotated_eval.json", "annotated_eval_species.csv"])